In [ ]:
pip install --upgrade --force-reinstall langchain langchain-community langchain-openai transformers sentence-transformers faiss-cpu pypdf fpdf

## 테스트(gemini)


### 한글 폰트 설치 및 설정

`fpdf`에서 한글을 사용하기 위해서는 한글 폰트가 필요합니다. `apt-get`을 통해 나눔고딕 폰트를 설치하고, 이를 `fpdf`에 등록하여 사용합니다.

In [ ]:
# 나눔고딕 폰트 설치
!apt-get update -qq > /dev/null
!apt-get install fonts-nanum* -qq > /dev/null

import matplotlib.font_manager as fm
import os

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

In [ ]:
# 예시를 위한 더미 PDF 파일 생성
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()

# 폰트 추가 및 설정 (한글 지원)
pdf.add_font('NanumGothic', '', '/usr/share/fonts/truetype/nanum/NanumGothic.ttf', uni=True)
pdf.set_font('NanumGothic', '', 12)

pdf.multi_cell(0, 10, """
안녕하세요. 이것은 RAG 기반 Q&A 시스템을 위한 예제 문서입니다.
문서를 작은 조각으로 나누는 것은 검색 효율성을 높이는 데 매우 중요합니다.

각 청크는 특정 크기와 오버랩을 가질 수 있으며, 이는 문맥을 유지하는 데 도움이 됩니다.
예를 들어, 텍스트 분할기는 문장을 기준으로 나눌 수도 있고, 특정 문자 수를 기준으로 나눌 수도 있습니다.

LangChain의 RecursiveCharacterTextSplitter는 다양한 구분자(예: \n\n, \n, 공백 등)를 사용하여 텍스트를 재귀적으로 분할합니다.
이는 복잡한 문서를 효과적으로 처리할 수 있게 해줍니다.

우리는 이 분할된 청크들을 임베딩으로 변환하여 벡터 데이터베이스에 저장할 것입니다.
그리고 사용자 질문이 들어오면, 이 벡터 데이터베이스에서 가장 유사한 청크들을 검색하여 답변 생성에 활용합니다.

이 과정은 Q&A 시스템의 정확성과 관련성을 향상시키는 핵심 단계입니다.
""")
pdf.output("sample_document.pdf")

print("더미 PDF 파일 'sample_document.pdf'가 생성되었습니다.")

### 1. PDF 문서 로드 및 텍스트 분할

`PyPDFLoader`를 사용하여 PDF 파일을 로드하고, `RecursiveCharacterTextSplitter`를 사용하여 문서를 청크로 분할합니다. `chunk_size`와 `chunk_overlap`은 중요하며, 이는 생성될 청크의 크기와 청크 간의 겹치는 부분을 정의합니다. 적절한 값은 사용하려는 데이터와 목적에 따라 달라질 수 있습니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF 파일 로드
loader = PyPDFLoader("sample_document.pdf")
documents = loader.load()

# 텍스트 분할기 설정
# chunk_size: 각 청크의 최대 문자 수
# chunk_overlap: 청크 간의 겹치는 문자 수 (문맥 유지를 위함)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 200자로 청크 분할
    chunk_overlap=20, # 20자 오버랩
    length_function=len,
    add_start_index=True,
)

# 문서 분할
splitted_documents = text_splitter.split_documents(documents)

print(f"원본 문서의 수: {len(documents)}")
print(f"분할된 청크의 수: {len(splitted_documents)}")
print("\n--- 첫 번째 청크 내용 ---")
print(splitted_documents[0].page_content)
print("\n--- 두 번째 청크 내용 ---")
print(splitted_documents[1].page_content)
print("\n--- 마지막 청크 내용 ---")
print(splitted_documents[-1].page_content)

### 2. 임베딩 생성 및 벡터 DB 저장

문서 청크들을 벡터 임베딩으로 변환하고, 이를 FAISS 벡터 데이터베이스에 저장합니다. 임베딩 모델은 문장의 의미를 벡터 공간에 표현하여 유사한 문장들이 가까운 위치에 있도록 합니다. FAISS는 대규모 벡터 데이터셋에서 효율적인 유사성 검색을 가능하게 합니다.

In [ ]:
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델 로드 (한국어 모델 사용)
# 'sentence-transformers/snunlp-KR-SBERT-V40K' 또는 'jhgan/ko-sroberta-multitask' 등을 사용할 수 있습니다.
# 여기서는 'jhgan/ko-sroberta-multitask'를 사용합니다.
# 로컬에 모델이 없으면 다운로드에 시간이 걸릴 수 있습니다.
embedding_model = SentenceTransformerEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# FAISS 벡터 데이터베이스 생성
# splitted_documents의 각 Document 객체를 임베딩하고 FAISS 인덱스를 생성합니다.
vector_store = FAISS.from_documents(splitted_documents, embedding_model)

print("FAISS 벡터 데이터베이스가 성공적으로 생성되었습니다.")
print(f"저장된 벡터의 수: {len(vector_store.docstore._dict)}")

In [ ]:
print("Attempting to resolve pydantic version conflict...")
!pip install --upgrade 'pydantic<2'
print("Pydantic version update complete. Please RESTART YOUR RUNTIME (Runtime -> Restart runtime) and then re-run all cells from the beginning.")

In [ ]:
print("Attempting to force-reinstall langchain and langchain-community to resolve module import issues...")
!pip install --upgrade --force-reinstall langchain langchain-community
print("Reinstallation attempt complete. Please re-run the subsequent cells to initialize the Q&A system.")

### 0. 필수 라이브러리 설치 및 의존성 해결

`langchain`과 관련 라이브러리, `fpdf`, `sentence-transformers` 등을 설치합니다. 특히 `pydantic` 버전 충돌 문제를 해결하기 위해 `pydantic<2`를 먼저 설치하고, `langchain` 관련 패키지를 `force-reinstall`하여 모든 의존성이 올바르게 로드되도록 합니다.

In [ ]:
print("Performing a clean reinstallation of required packages with specific langchain version...")

# Explicitly uninstall potentially conflicting packages before re-installation
# This ensures a clean slate, removing any partially installed or corrupted files.
!pip uninstall -y pydantic langchain langchain-community langchain-openai transformers sentence-transformers faiss-cpu pypdf fpdf --quiet

# Then perform a fresh, forced installation of all necessary packages, with specific langchain version
!pip install --upgrade --force-reinstall langchain==0.1.20 langchain-community langchain-openai transformers sentence-transformers faiss-cpu pypdf fpdf --quiet

print("Clean reinstallation with specific langchain version complete. Please RESTART YOUR RUNTIME (Runtime -> Restart runtime) and then re-run all cells from the beginning.")


### 3. 임베딩 생성 및 벡터 DB 저장

문서 청크들을 벡터 임베딩으로 변환하고, 이를 FAISS 벡터 데이터베이스에 저장합니다.

In [ ]:
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델 로드 (한국어 모델 사용)
embedding_model = SentenceTransformerEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# FAISS 벡터 데이터베이스 생성
vector_store = FAISS.from_documents(splitted_documents, embedding_model)

print("FAISS 벡터 데이터베이스가 성공적으로 생성되었습니다.")
print(f"저장된 벡터의 수: {len(vector_store.docstore._dict)}")

### 4. Ollama를 이용한 질문-답변 시스템 구축

Ollama 모델을 사용하여 질문에 대한 답변을 생성하는 시스템을 구축합니다. 이를 위해서는 로컬에 Ollama 서버가 실행 중이어야 합니다.

In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain

# Ollama LLM 모델 초기화
# 로컬에 Ollama 서버가 실행 중이며, 'llama2' 또는 원하는 모델이 pull 되어 있어야 합니다.
llm = ChatOllama(model="llama2") # 예시: llama2 모델 사용. 다른 모델로 변경 가능.

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """당신은 제공된 문맥을 바탕으로 질문에 답변하는 유용한 AI 비서입니다.
    질문에 답변하기 위해 다음 검색된 문맥만 사용하십시오.
    만약 문맥에서 답을 찾을 수 없다면, 모른다고 말하고 추측하지 마십시오.
    답변은 한국어로 제공하십시오.

    {context}

    질문: {input}"""
)

# 문서 체인 생성
document_chain = create_stuff_documents_chain(llm, prompt)

# 검색기를 포함한 전체 검색 체인 생성
retriever = vector_store.as_retriever()
qa_chain = create_retrieval_chain(retriever, document_chain)

print("Q&A 시스템이 Ollama를 사용하여 성공적으로 초기화되었습니다.")

In [ ]:
import os
import langchain

# Get the base path of the langchain installation
langchain_path = os.path.dirname(langchain.__file__)
chains_path = os.path.join(langchain_path, 'chains')

print(f"LangChain 설치 경로: {langchain_path}")
print(f"langchain.chains 디렉토리 확인 중: {chains_path}")

if os.path.exists(chains_path) and os.path.isdir(chains_path):
    print(f"'chains' 디렉토리가 존재합니다: {chains_path}")
    print(f"'chains' 디렉토리 내용 (일부): {os.listdir(chains_path)[:5]}...")
    print("이는 'langchain.chains' 모듈이 물리적으로 존재함을 의미합니다. 문제는 임포트 경로 또는 환경 설정에 있을 수 있습니다.")
else:
    print(f"경고: 'chains' 디렉토리가 발견되지 않았습니다: {chains_path}")
    print("이는 'langchain' 설치에 문제가 있음을 나타냅니다. 'langchain.chains' 모듈 파일이 누락된 것으로 보입니다.")


### 5. 질문 입력 및 답변 받기

이제 사용자로부터 질문을 입력받아 `qa_chain`을 통해 답변을 생성해볼 수 있습니다.

In [ ]:
if qa_chain:
    # 사용자 질문 입력
    question = input("질문을 입력하세요: ")

    # 답변 생성
    response = qa_chain.invoke({"input": question})

    print("\n--- 답변 --- ")
    print(response["answer"])
else:
    print("Q&A 시스템이 초기화되지 않아 질문에 답변할 수 없습니다.")

## BGE-M3 모델 테스트

여기서부터 개인 실습


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
model = SentenceTransformer("BAAI/bge-m3")

In [ ]:
#유사도 테스트

# 약품 관련 예시 문장들
sentences = [
    "해열 진통제 타이레놀 복용법",          # 기준 문장 (아세트아미노펜)
    "열이 날 때 먹는 어린이 해열제 안내",    # 유사 증상/목적
    "아세트아미노펜 성분의 진통제 종류",    # 성분 기반 유사성
    "바르는 근육통 파스 사용 시 주의사항",  # 외용제 (비교적 낮은 유사도)
    "오메가3 영양제 권장 섭취량"           # 전혀 다른 카테고리
]

vectors = model.encode(sentences)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

base_vector = vectors[0]

print(f"기준 문장: {sentences[0]}\n")
print(f"{'유사도':<6} | {'비교 문장'}")
print("-" * 50)

for i, s in enumerate(sentences):
    score = cosine_similarity(base_vector, vectors[i])
    print(f"{score:.4f} | {s}")
#------------------------------------------------------------
sentences2 = [
    "타이레놀 500mg 복용 방법과 주의사항",
    "아세트아미노펜 500밀리그램 섭취 시 유의점",
    "두통약 타이레놀 먹는 법과 부작용 안내",
    "Tylenol 500mg dosage and precautions",
    "해열제 복용 시 지켜야 할 사항들",
    "타이레놀 400mg 복용 방법과 주의사항",
    "타이레놀 500밀리그램 섭취 시 유의점"
]

vectors2 = model.encode(sentences2)
base_vector_2 = vectors2[0]


print(f"\n기준 문장: {sentences2[0]}\n")
print(f"{'유사도':<6} | {'비교 문장'}")
print("-" * 50)

for i, s in enumerate(sentences2):
    score2 = cosine_similarity(base_vector_2, vectors2[i])
    print(f"{score2:.4f} | {s}")

In [ ]:
#유사도 테스트 기반 QnA 샘플

# 2. 지식 베이스 (질문-답변 쌍)
faq_database = [
    {
        "q": "탁센 이브는 어떤 증상에 먹나요? 효능이 궁금해요.",
        "a": "두통, 치통, 생리통(월경통), 근육통, 관절통, 요통 등 각종 통증의 진통과 오한, 발열 시 해열 효과가 있습니다."
    },
    {
        "q": "성인은 하루에 몇 알까지 먹을 수 있나요? 복용량 알려주세요.",
        "a": "만 15세 이상 성인은 1일 1~3회, 1회 1~2캡슐을 복용합니다. 복용 간격은 최소 4시간 이상이어야 하며, 공복을 피해서 복용하세요."
    },
    {
        "q": "초등학생이나 중학생 아이가 먹어도 되나요?",
        "a": "만 11세~15세 미만은 1회 1캡슐(1일 3회까지), 만 8세~11세 미만은 1회 1캡슐(1일 2회까지) 복용 가능합니다. 만 8세 미만은 복용 전 의사·약사와 상의해야 하며, 만 2세 미만은 투여하지 않습니다."
    },
    {
        "q": "빈속에 먹어도 괜찮나요?",
        "a": "위장 장애를 줄이기 위해 가급적 공복(빈 속) 시를 피하여 복용하는 것이 좋습니다."
    },
    {
        "q": "술을 마셨는데 먹어도 될까요?",
        "a": "매일 3잔 이상 정기적으로 술을 마시는 사람이 복용할 경우 위장 출혈이 유발될 수 있습니다. 복용 시에는 음주를 절대 피해야 합니다."
    },
    {
        "q": "임신 중인데 복용해도 안전한가요?",
        "a": "임신 20주 이후에는 태아의 신장 이상이나 양수 과소증을 일으킬 수 있으며, 특히 30주 이후에는 태아의 동맥관 조기 폐쇄 위험이 있어 복용이 권고되지 않습니다. 반드시 의사와 상의하세요."
    },
    {
        "q": "다른 감기약이나 진통제랑 같이 먹어도 되나요?",
        "a": "이 약을 복용하는 동안 다른 해열진통제, 감기약, 진정제를 함께 복용해서는 안 됩니다."
    },
    {
        "q": "심장이 안 좋거나 혈압이 높은데 주의할 점이 있나요?",
        "a": "고혈압, 심부전, 허혈성 심장질환이 있는 경우 신중히 투여해야 합니다. 고용량(1일 2400mg 이상) 복용 시 심근경색이나 뇌졸중 위험이 증가할 수 있으므로 주의가 필요합니다."
    },
    {
        "q": "부작용에는 어떤 것들이 있나요?",
        "a": "발진, 가려움, 구역, 구토, 소화관 출혈, 어지러움, 부종 등이 나타날 수 있습니다. 드물게 아나필락시스 쇼크나 심각한 피부 반응이 나타나면 즉시 복용을 중단해야 합니다."
    },
    {
        "q": "약은 어떻게 보관해야 하나요?",
        "a": "어린이의 손이 닿지 않는 곳에, 직사광선을 피해 습기가 적고 서늘한 곳에 밀폐하여 보관하세요. 원래 용기 그대로 보관하는 것이 품질 유지에 좋습니다."
    }
]

# 3. 질문들만 임베딩 (미리 계산해두면 빠름)
questions = [item['q'] for item in faq_database]
question_embeddings = model.encode(questions, convert_to_tensor=True)

def ask_qna(user_query):
    # 사용자 질문 임베딩
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # 유사도 계산
    hits = util.semantic_search(query_embedding, question_embeddings, top_k=1)

    # 결과 반환 (유사도가  0.6 이상일 때만 답변)
    if hits[0][0]['score'] > 0.6:
        index = hits[0][0]['corpus_id']
        return faq_database[index]['a']
    else:
        return "죄송합니다. 답변을 찾지 못했습니다."
#아예 a랑 관련되게 해서 제일 유사도 높은 답변 출력하게 하는 것도 좋을듯. 내일 해보기로.
print(ask_qna("혈압 높은데 주의할 점 있나요?"))

In [ ]:
!pip install ollama


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

answer_corpus = [
    "효능 및 효과: 두통, 치통, 생리통, 근육통, 신경통, 요통, 관절통 등 각종 통증 완화와 해열.",
    "용법 용량(성인): 만 15세 이상 성인은 1일 1~3회, 1회 1~2캡슐을 4시간 이상 간격으로 공복을 피하여 복용하세요.",
    "소아 복용법: 만 11세~15세 미만은 1회 1캡슐(1일 3회), 만 8세~11세 미만은 1회 1캡슐(1일 2회) 복용합니다.",
    "금기 사항: 만 2세 미만 영아, 이 약에 과민증이 있는 사람, 천식 병력이 있는 사람은 복용하지 마십시오.",
    "음주 주의: 복용 시 음주는 금물입니다. 매일 3잔 이상 음주 시 위장 출혈이 발생할 수 있습니다.",
    "임부 주의: 임신 20주 이후에는 태아 신장 이상 위험이, 30주 이후에는 태아 동맥관 폐쇄 위험이 있어 복용을 피해야 합니다.",
    "병용 금지: 다른 해열진통제, 감기약, 진정제와 함께 복용하지 마십시오.",
    "부작용 및 이상반응: 발진, 가려움, 구역, 구토, 소화관 출혈, 부종 등이 나타나면 즉시 중단하고 전문가와 상의하세요."
]

# 3. 답변들을 임베딩 (인덱싱)
answer_embeddings = model.encode(answer_corpus, convert_to_tensor=True)

def get_answer(user_query):
    # 질문 임베딩
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # 질문과 모든 답변 후보들 간의 코사인 유사도 계산
    cos_scores = util.cos_sim(query_embedding, answer_embeddings)[0]

    # 가장 높은 점수의 결과 추출
    top_results = torch.topk(cos_scores, k=1)
    score = top_results.values[0].item()
    index = top_results.indices[0].item()

    # 유사도 임계값 설정
    if score > 0.45:
        return {
            "답": answer_corpus[index],
            "유사도": f"{score:.2f}"
        }
    else:
        return {"answer": "해당 내용에 대한 정보를 찾을 수 없습니다. 약사나 의사와 상의하세요.", "confidence": f"{score:.2f}"}

# 테스트
#print(f"질문: 임신했는데 먹어도 돼?\n답변: {get_answer('임신했는데 먹어도 돼?')}\n")
#print(f"질문: 술 마시고 먹으면?\n답변: {get_answer('술 마시고 먹으면?')}\n")
question = input()
print(f"\n{get_answer(question)}\n")